Just a simple cell to check if everything is well installed 

In [22]:
import sys
import pandas as pd
import numpy as np
import duckdb
import requests
from bs4 import BeautifulSoup
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import pyarrow
import openpyxl
import mlxtend
import ucimlrepo

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("duckdb:", duckdb.__version__)
print("requests:", requests.__version__)
print("beautifulsoup4: OK")
print("matplotlib:", plt.matplotlib.__version__)
print("seaborn:", sns.__version__)
print("scikit-learn:", sklearn.__version__)
print("pyarrow:", pyarrow.__version__)
print("openpyxl:", openpyxl.__version__)
print("mlxtend:", mlxtend.__version__)
print("ucimlrepo: OK")

Python: 3.13.5
pandas: 2.3.3
numpy: 2.5.1
duckdb: 1.5.4
requests: 2.34.2
beautifulsoup4: OK
matplotlib: 3.11.1
seaborn: 0.13.2
scikit-learn: 1.9.0
pyarrow: 25.0.1
openpyxl: 3.1.5
mlxtend: 0.23.4
ucimlrepo: OK


## Acquisition strategy


The acquisition has two stages:

1. `/discover/movie` is used to retrieve movies matching the registered scope:
   - release date: 2020-01-01 to 2025-12-31
   - origin country: US
   - genre: Drama OR Thriller
   - minimum vote count: 100

2. `/movie/{movie_id}/credits` is queried for every selected movie to retrieve:
   - director(s)
   - producer(s)
   - cinematographer(s)
   - top-6 billed actors


In [23]:
import os
import json
import time
import random
import hashlib
from pathlib import Path
from email.utils import parsedate_to_datetime
from datetime import datetime, timezone

import requests
import pandas as pd

from dotenv import load_dotenv

load_dotenv()

URL = "https://api.themoviedb.org/3"
DISCOVER = f"{URL}/discover/movie"

TMDB_TOKEN = os.getenv("TMDB_TOKEN")

assert TMDB_TOKEN, "TMDB_TOKEN is missing from .env"

HEADERS = {
    "Authorization": f"Bearer {TMDB_TOKEN}",
    "Accept": "application/json"
}

CACHE = Path("cache")
CACHE.mkdir(exist_ok=True)


print("TMDB credentials loaded ")
print("Token available:", bool(TMDB_TOKEN))
print("Token length:", len(TMDB_TOKEN))

TMDB credentials loaded 
Token available: True
Token length: 239


In [24]:
PARAMS = {
    "primary_release_date.gte": "2020-01-01",
    "primary_release_date.lte": "2025-12-31",
    "with_origin_country": "US",
    "with_genres": "18|53",
    "vote_count.gte": 100,
    "include_adult": "false",
    "include_video": "false",
    "language": "en-US"
}

def cache_key(url, params=None):
    payload = url + "|" + json.dumps(params or {}, sort_keys=True)
    return hashlib.sha256(payload.encode()).hexdigest()[:16]

def http_get(url, params=None, headers=HEADERS, attempts=5):
    for k in range(attempts):

        try:
            r = requests.get(url,params=params,headers=headers,timeout=10)

        except requests.RequestException as exc:
            status, retry_after = repr(exc), 0

        else:

            if r.status_code == 200:
                return r.json()

            if r.status_code not in (429, 500, 502, 503, 504):
                r.raise_for_status()

            # Temporary error -> retry
            status = r.status_code

            try:
                retry_after = int(r.headers.get("Retry-After", 0))
            except ValueError:
                retry_after = 0

        if k + 1 < attempts:
            wait = retry_after or random.uniform(0.001, 2**k)

            print(
                f"{status} on attempt {k + 1}, "
                f"sleeping {wait:.1f}s"
            )

            time.sleep(wait)

    raise RuntimeError(
        f"gave up after {attempts} attempts: {url}"
    )


def cached_get(url, params=None):
    path = CACHE / f"{cache_key(url, params)}.json"

    if path.exists():
        return json.loads(path.read_text())

    body = http_get(url, params)

    path.write_text(
        json.dumps(body)
    )

    return body

In [25]:
## test if it works 
params = PARAMS.copy()
params["page"] = 1

payload = cached_get(
    DISCOVER,
    params
)

print("page:", payload["page"])
print("total pages:", payload["total_pages"])
print("total results:", payload["total_results"])
print("results on page:", len(payload["results"]))

page: 1
total pages: 61
total results: 1204
results on page: 20


In [27]:
##getting all movies from the API

movies = []
page = 1
total_pages = None

while True:

    params = PARAMS.copy()
    params["page"] = page

    payload = cached_get(
        DISCOVER,
        params
    )

    movies += payload["results"]

    if total_pages is None:
        total_pages = payload["total_pages"]

    print(
        f"page {page}/{total_pages} "
        f"- {len(movies)} movies"
    )

    if page >= total_pages:
        break

    page += 1

page 1/61 - 20 movies
page 2/61 - 40 movies
page 3/61 - 60 movies
page 4/61 - 80 movies
page 5/61 - 100 movies
page 6/61 - 120 movies
page 7/61 - 140 movies
page 8/61 - 160 movies
page 9/61 - 180 movies
page 10/61 - 200 movies
page 11/61 - 220 movies
page 12/61 - 240 movies
page 13/61 - 260 movies
page 14/61 - 280 movies
page 15/61 - 300 movies
page 16/61 - 320 movies
page 17/61 - 340 movies
page 18/61 - 360 movies
page 19/61 - 380 movies
page 20/61 - 400 movies
page 21/61 - 420 movies
page 22/61 - 440 movies
page 23/61 - 460 movies
page 24/61 - 480 movies
page 25/61 - 500 movies
page 26/61 - 520 movies
page 27/61 - 540 movies
page 28/61 - 560 movies
page 29/61 - 580 movies
page 30/61 - 600 movies
page 31/61 - 620 movies
page 32/61 - 640 movies
page 33/61 - 660 movies
page 34/61 - 680 movies
page 35/61 - 700 movies
page 36/61 - 720 movies
page 37/61 - 740 movies
page 38/61 - 760 movies
page 39/61 - 780 movies
page 40/61 - 800 movies
page 41/61 - 820 movies
page 42/61 - 840 movies
page 

In [28]:
##normalize the data 
movie_columns = [
    "id",
    "title",
    "original_title",
    "release_date",
    "genre_ids",
    "original_language",
    "vote_count",
    "vote_average",
    "popularity"
]

movies_df = pd.json_normalize(movies) ## all the distinct movies in the dataset
movies_df = movies_df[movie_columns]
movies_genre = movies_df.explode("genre_ids") ## (movie, genre) per row


movies_genre

,id,title,original_title,release_date,genre_ids,original_language,vote_count,vote_average,popularity
0,755898,War of the Worlds,War of the Worlds,2025-07-29,878,en,1147,4.028,87.7017
0,755898,War of the Worlds,War of the Worlds,2025-07-29,53,en,1147,4.028,87.7017
1,591634,False Positive,False Positive,2021-06-18,27,en,200,5.500,56.2980
1,591634,False Positive,False Positive,2021-06-18,9648,en,200,5.500,56.2980
1,591634,False Positive,False Positive,2021-06-18,53,en,200,5.500,56.2980
...,...,...,...,...,...,...,...,...,...
1202,728874,Violation,Violation,2020-09-14,18,en,138,5.442,1.4051
1203,848278,Jurassic Hunt,Jurassic Hunt,2021-08-26,28,en,221,5.016,1.2990
1203,848278,Jurassic Hunt,Jurassic Hunt,2021-08-26,878,en,221,5.016,1.2990
1203,848278,Jurassic Hunt,Jurassic Hunt,2021-08-26,53,en,221,5.016,1.2990


In [29]:

CREDITS_PARAMS = {
    "language": "en-US"
}

JOBS = {
    "Director": "director",
    "Producer": "producer",
    "Director of Photography": "cinematographer"
}

credits_element= []

for movie_id in movies_df["id"].unique():

    credits_url = f"{URL}/movie/{movie_id}/credits"

    credits = cached_get(credits_url,CREDITS_PARAMS)

    for person in credits["crew"]:

        if person["job"] in JOBS:

            credits_element.append({
                "movie_id": movie_id,
                "person_id": person["id"],
                "person_name": person["name"],
                "role": JOBS[person["job"]],
                "department": person["department"],
                "job": person["job"],
                "billing_order": None
            })


    top_actors = sorted(credits["cast"],key=lambda actor: actor["order"])[:6]

    for actor in top_actors:

        credits_element.append({
            "movie_id": movie_id,
            "person_id": actor["id"],
            "person_name": actor["name"],
            "role": "actor",
            "department": None,
            "job": None,
            "billing_order": actor["order"]
        })

In [30]:
credits_df = pd.json_normalize(credits_element)
print("Credits DataFrame shape:", credits_df.shape)

display(credits_df)

Credits DataFrame shape: (14207, 7)


,movie_id,person_id,person_name,role,department,job,billing_order
0,755898,964093,Patrick Aiello,producer,Production,Producer,NaN
1,755898,23430,Timur Bekmambetov,producer,Production,Producer,NaN
2,755898,1634181,Rich Lee,director,Directing,Director,NaN
3,755898,1076356,Christopher Probst,cinematographer,Camera,Director of Photography,NaN
4,755898,9778,Ice Cube,actor,None,None,0.0
...,...,...,...,...,...,...,...
14202,848278,1186714,Ruben Pla,actor,None,None,1.0
14203,848278,1875337,Tarkan Dospil,actor,None,None,2.0
14204,848278,1299438,Joston Theney,actor,None,None,3.0
14205,848278,2052540,Antuone Torbert,actor,None,None,4.0
